In [ ]:
# （1）MCP 客户端：mcp SDK v2 自带高层 Client，无需自己封装传输和握手
# async with Client(...) 进入时自动完成 initialize 握手，退出时自动断开；
# 握手后核心 API 只有两个：list_tools()（tools/list 发现工具）、call_tool()（tools/call 调用工具）

from importlib.metadata import version
from mcp import Client

print("MCP 客户端 = mcp.Client，mcp SDK 版本 =", version("mcp"))


MCP 客户端 = mcp.Client，mcp SDK 版本 = 2.3.0


In [6]:
# （2）本地天气 MCP 服务端（mock 数据）：@server.tool() 把普通函数注册成 MCP 工具

from datetime import date
from pydantic import BaseModel
from mcp.server import MCPServer

MOCK = {"北京": ("晴", 25, 40), "上海": ("多云", 22, 65), "广州": ("小雨", 28, 80)}

class Weather(BaseModel):
    city: str
    date: str
    condition: str
    temp_c: int
    humidity_pct: int

server = MCPServer("local-weather", version="1.0.0")

# 注册MCP工具：查询指定城市当天的天气（mock 数据）
@server.tool()
def get_weather(city: str) -> Weather:
    """查询指定城市当天的天气（mock 数据）"""
    condition, temp_c, humidity = MOCK.get(city, ("未知", 0, 0))
    return Weather(city=city, date=str(date.today()), condition=condition, temp_c=temp_c, humidity_pct=humidity)

async def weather_search_local():
    # MCP客户端连接本地服务器
    async with Client(server) as client:  
        print(f"服务端: {client.server_info.name} v{client.server_info.version} / 协议 {client.protocol_version}")
        for t in (await client.list_tools()).tools:
            print(f"发现工具 {t.name}: {(t.description or '').splitlines()[0]}")

        result = await client.call_tool("get_weather", {"city": "北京"}) # 调用工具并返回结果
        print(f"\ncall_tool(get_weather, city='北京') ")
        print(f"返回值 = {result.structured_content}")

await weather_search_local()  
# notebook 顶层可直接 await；.py 脚本里写 asyncio.run(weather_search_local())

服务端: local-weather v1.0.0 / 协议 2026-07-28
发现工具 get_weather: 查询指定城市当天的天气（mock 数据）

call_tool(get_weather, city='北京') 
返回值 = {'city': '北京', 'date': '2026-10-22', 'condition': '晴', 'temp_c': 25, 'humidity_pct': 40}


In [11]:
# （3）远程开源天气 MCP 服务：把本地的「MCPServer 实例」换成 URL（streamable-http）

# 开源天气 MCP 服务端
REMOTE_WEATHER_URL = "https://www.modelmarket.dev/tools/mcp/weather"

async def weather_search_remote():
    # MCP客户端连接远程服务端器
    async with Client(REMOTE_WEATHER_URL) as client:
        print(f"服务端: {client.server_info.name} v{client.server_info.version} / 协议 {client.protocol_version}")
        for t in (await client.list_tools()).tools:
            print(f"发现工具 {t.name}: {(t.description or '').splitlines()[0]}")

        result = await client.call_tool("weather_now", {"city": "北京"}) # 调用工具并返回结果
        reading = result.structured_content["result"]["reading"]
        print(f"\nweather_now(city='北京') ")
        print(f"  地点/时间 = {result.structured_content['result']['resolved']['matched_place']} @ {reading['ts']}")
        print(f"  返回值    = {reading['values']}")


await weather_search_remote()

[10/22/26 22:31:27] INFO     HTTP Request: POST https://www.modelmarket.dev/tools/mcp/weather       ]8;id=5941588;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=5941589;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             "HTTP/1.1 200 OK"                                                                     

                    INFO     HTTP Request: POST https://www.modelmarket.dev/tools/mcp/weather       ]8;id=5941594;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=5941595;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             "HTTP/1.1 200 OK"                                                                     

服务端: modelmarket-weather v1.0.0 / 协议 2025-11-25


[10/22/26 22:31:28] INFO     HTTP Request: POST https://www.modelmarket.dev/tools/mcp/weather       ]8;id=5941600;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=5941601;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             "HTTP/1.1 202 Accepted"                                                               

                    INFO     HTTP Request: POST https://www.modelmarket.dev/tools/mcp/weather       ]8;id=5941606;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=5941607;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             "HTTP/1.1 200 OK"                                                                     

发现工具 weather_now: Current weather at a place: temperature (°C), humidity (%), pressure (hPa) and wind (m/s) from the nearest live Open-Meteo relay within 75 km, with a signed receipt. Send latitude and longitude, or a city.
发现工具 air_quality_now: Current air quality at a place: PM2.5 and PM10 (µg/m³), US and European AQI, from Copernicus CAMS via Open-Meteo, with a signed receipt. Send latitude and longitude, or a city.
发现工具 nearby_sensors: The nearest live public sensors to a point, one per layer asked for (weather, air, radiation, quake …), each with its reading, distance and source, and a signed receipt.


[10/22/26 22:31:29] INFO     HTTP Request: POST https://www.modelmarket.dev/tools/mcp/weather       ]8;id=5941612;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=5941613;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             "HTTP/1.1 200 OK"                                                                     


weather_now(city='北京') 
  地点/时间 = Beijing @ 2026-10-22T14:31:26Z
  返回值    = {'temperature_c': 9.6, 'humidity_pct': 59, 'pressure_hpa': 1016.5, 'wind_mps': 2.71}


In [ ]:
#  LLM + langchain + MCP：让 LLM 调用 MCP 工具查询天气

import os
import json
from pathlib import Path
from configparser import ConfigParser

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# notebook cell 里没有 __file__，用 cwd 定位项目根目录（cwd 就是 demo/）
BASE_PATH = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "demo" else os.getcwd()
config_file = os.path.join(BASE_PATH, "config.ini")
config = ConfigParser()
config.read(config_file, encoding='utf-8')

api_key = os.environ.get("LLM_API_KEY")
if not api_key:
    raise ValueError("环境变量 LLM_API_KEY 未设置")

llm = ChatOpenAI(
    api_key=api_key,
    base_url=config.get("LLM", "api_url"),
    model=config.get("LLM", "model_name"),
    temperature=float(config.get("LLM", "temperature")),
    max_tokens=512
)

@tool
async def get_weather(city: str) -> str:
    """查询指定城市的实时天气"""
    async with Client(REMOTE_WEATHER_URL) as client:
        result = await client.call_tool("weather_now", {"city": city})
    if result.is_error:
        return json.dumps({"城市": city, "错误": result.content[0].text[:200]}, ensure_ascii=False)
    values = result.structured_content["result"]["reading"]["values"]
    return json.dumps(
        {"城市": city, "温度°C": values["temperature_c"],
         "湿度%": values["humidity_pct"], "风速m/s": values["wind_mps"]},
        ensure_ascii=False,
    )

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="你是天气助手。需要天气数据时必须调用工具，最后汇总对比。",
)

async def weather_search_llm():
    # 让 LLM 调用 MCP 工具查询天气并对比
    response = await agent.ainvoke(
        {"messages": [("user", "查询北京、上海、深圳三地天气并对比")]},
        config={"recursion_limit": 10},
    )
    for message in response["messages"]:
        if message.type == "ai" and message.tool_calls:
            print("LLM 决定调用:", [(t["name"], t["args"]) for t in message.tool_calls])
        elif message.type == "tool":
            print("工具返回:", message.content)
    print("\nAgent 汇总:", response["messages"][-1].content)

await weather_search_llm()

[10/22/26 22:56:58] INFO     HTTP Request: POST                                                     ]8;id=5941727;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=5941728;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions                    
                             "HTTP/1.1 200 OK"                                                                     

[10/22/26 22:56:59] INFO     HTTP Request: POST https://www.modelmarket.dev/tools/mcp/weather       ]8;id=5941733;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=5941734;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[10/22/26 22:57:00] INFO     HTTP Request: POST https://www.modelmarket.dev/tools/mcp/weather       ]8;id=5941739;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=5941740;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             "HTTP/1.1 200 OK"                                                                     

                    INFO     HTTP Request: POST https://www.modelmarket.dev/tools/mcp/weather       ]8;id=5941745;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=5941746;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             "HTTP/1.1 200 OK"                                                                     

                    INFO     HTTP Request: POST https://www.modelmarket.dev/tools/mcp/weather       ]8;id=5941751;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=5941752;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             "HTTP/1.1 202 Accepted"                                                               

...

[10/22/26 22:57:04] INFO     HTTP Request: POST https://www.modelmarket.dev/tools/mcp/weather       ]8;id=5941805;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=5941806;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[10/22/26 22:57:05] INFO     HTTP Request: POST https://www.modelmarket.dev/tools/mcp/weather       ]8;id=5941811;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=5941812;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[10/22/26 22:57:06] INFO     HTTP Request: POST https://www.modelmarket.dev/tools/mcp/weather       ]8;id=5941817;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=5941818;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[10/22/26 22:57:11] INFO     HTTP Request: POST                                                     ]8;id=5941823;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py\_client.py]8;;\:]8;id=5941824;file://d:\miniconda\envs\agent\Lib\site-packages\httpx2\_client.py#1922\1922]8;;\
                             https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions                    
                             "HTTP/1.1 200 OK"                                                                     

LLM 决定调用: [('get_weather', {'city': '北京'}), ('get_weather', {'city': '上海'}), ('get_weather', {'city': '深圳'})]
工具返回: {"城市": "北京", "温度°C": 9.4, "湿度%": 60, "风速m/s": 2.6}
工具返回: {"城市": "上海", "温度°C": 17.7, "湿度%": 66, "风速m/s": 2.2}
工具返回: {"城市": "深圳", "温度°C": 24.2, "湿度%": 75, "风速m/s": 1.55}

Agent 汇总: 以下是北京、上海、深圳三地的天气对比：

| 城市 | 温度 (°C) | 湿度 (%) | 风速 (m/s) |
|------|-----------|----------|------------|
| 北京 | 9.4       | 60       | 2.6        |
| 上海 | 17.7      | 66       | 2.2        |
| 深圳 | 24.2      | 75       | 1.55       |

**简要分析：**  
- **温度**：深圳最暖（24.2°C），上海次之（17.7°C），北京最凉（9.4°C），呈现明显的南暖北冷趋势。  
- **湿度**：深圳最高（75%），北京最低（60%），南方空气更湿润。  
- **风速**：北京风力最强（2.6 m/s），深圳最弱（1.55 m/s）。  

如需进一步分析（如体感温度、穿衣建议或未来预报），欢迎随时告诉我！
